# 🎨 Notebook 1: Car Rental — Class Design


## 🛠️ Setup

```bash
cd 07-object-oriented-design/car-rental
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This notebook is **all design, no code**. It explains *why* we model the
> system the way we do. The runnable code lives in `02_implementation.ipynb`.


## 🧭 What problem are we solving?

Imagine **Turo** or **Hertz** in miniature. A customer picks dates and a
vehicle, we check that the car is free on those dates, we take payment, the
customer picks up the car, drives it, returns it, and we charge late fees
if they bring it back a day late.

Sounds simple — but a good design has to handle tricky things:

- The **same car** can be rented by Alice next week and Bob the week after.
  We can't just put `is_available = True/False` on the car: that flag is
  true or false depending on *which dates* you ask about.
- Prices are not one number. Weekend days cost more, long rentals get a
  discount, insurance and a GPS add-on cost extra.
- A reservation is not just a row — it moves through **stages**
  (booked → picked up → returned) and illegal jumps must be rejected
  (e.g. you can't "return" a car you never picked up).
- A company has **multiple branches**, and a customer might pick up in
  one city and drop off in another.

A good class design makes each of these things land in *exactly one* place.


## 👥 Actors & use cases

| Actor | Wants to… |
|---|---|
| **Customer** | search available cars on dates, book, pay, pick up, return |
| **Rental agent** | confirm pickups & returns, inspect the car, charge late fees |
| **Admin** | add vehicles to the fleet, set prices, open new branches |
| **System** | prevent double-booking, compute totals, move reservations through stages |

Main use cases we'll design for:

1. **Search** — "Any SUV in Tel Aviv for Fri–Sun?"
2. **Book** — Reserve that SUV, take a payment.
3. **Pick up / return** — Move the reservation through its lifecycle.
4. **Late return** — Charge an extra day + penalty.


## 🗺️ Domain model (at a glance)

```
 ┌──────────┐            ┌─────────────┐       ┌──────────┐
 │ Customer │── makes ──▶│ Reservation │──for──▶│ Vehicle  │ (abstract)
 └──────────┘            └─────────────┘        └────┬─────┘
                              │   │                  ▲
                              │   │has a        ┌────┼────┬────┬─────┐
                              │   ▼            Car  SUV  Van  Truck
                              │ ┌─────────┐
                              │ │ Payment │
                              │ └─────────┘
                              │
                              │ lifecycle
                              ▼
                    PENDING → CONFIRMED → ACTIVE → RETURNED
                                          │
                                          └──▶ CANCELLED
```

- **Vehicle** is an abstract base. `Car`, `SUV`, `Van`, `Truck` set their own
  price and seat count (that is **polymorphism**).
- **Reservation** links a `Customer` to a `Vehicle` for a date range and
  tracks which stage it is in (the **State** pattern, in its simplest form).
- **Branch** (location) owns a fleet of vehicles — a customer picks up at
  one branch.
- **PricingPolicy** decides the total (weekend multiplier, long-rental
  discount). That is the **Strategy** pattern.


## 🧱 Class responsibilities

| Class | Holds / knows | Does |
|---|---|---|
| `Customer` | id, name, license | (data only) |
| `Vehicle` (abstract) | plate | `daily_rate()`, `seats()` |
| `Car / SUV / Van / Truck` | — | concrete rate + seats |
| `Reservation` | customer, vehicle, start/end, state | `confirm`, `pick_up`, `drop_off`, `cancel`, `overlaps`, `days` |
| `Payment` | amount, method, status | `charge`, `refund` |
| `PricingPolicy` | — | `price(vehicle, start, end) → $` |
| `Branch` | name, fleet | `available(s, e, category=None)` |
| `RentalStore` | branches, pricing | `search`, `book`, `pick_up`, `drop_off`, `cancel` |

**Heuristic used above:** if one sentence describes a class with an "and"
in the middle — *"it knows the price and also talks to the database"* —
that is usually two classes.


## 🧠 Key design decisions (and why)

### 1. Availability from **overlapping dates**, not a boolean

Bad idea ❌:

```python
class Vehicle:
    is_available: bool   # can't answer "available next Friday?"
```

Good idea ✅: ask the list of reservations *"does any active reservation
for this car overlap the dates I care about?"*. Two date ranges `[a,b]`
and `[c,d]` **overlap** iff `not (b < c or a > d)`.

### 2. Rate is a **method**, not a field on a base class

We could put `daily_rate: float` on `Vehicle` and set it per instance, but
then pricing logic (weekend surcharge, long-stay discount) would leak into
every caller. Putting it behind `daily_rate()` lets subclasses — or a
`PricingPolicy` — change the rule in one place.

### 3. Reservation has **explicit states**, not flags

`is_confirmed`, `is_active`, `is_returned` booleans drift out of sync. A
single `state` field plus guarded transitions
(`confirm()`, `pick_up()`, `drop_off()`) makes illegal moves impossible.

### 4. Pricing is a **Strategy**, not a hard-coded formula

`FlatPricing`, `WeekendWeeklyPricing`, `LoyaltyPricing` — swap at runtime
without touching `Reservation`.


## 🧩 Patterns you'll see in Notebook 2

- **Polymorphism** — `Vehicle.daily_rate()` overridden per subtype.
- **State** (lightweight) — `ReservationState` enum + guarded transitions.
- **Strategy** — `PricingPolicy` plugs into `RentalStore.book`.
- **Factory-ish** — creating the right `Vehicle` subclass from fleet data.

You don't need to memorize pattern names. Notice instead *the shape*:
"one thing varies, everything else stays the same → isolate the thing that
varies behind an interface".


## ⚖️ Trade-offs (what we are NOT doing)

- **In-memory only.** Real systems store reservations in a DB with a
  unique constraint that prevents double-booking under concurrency. Our
  `available()` check is racy — fine for learning, not for production.
- **One price per vehicle type.** Real fleets price per-vehicle,
  per-season, per-customer-tier. The Strategy class makes that easy to add.
- **No user auth, no notifications, no receipts, no taxes.** On purpose —
  scope control.

## ✅ Ready?

Jump to [`02_implementation.ipynb`](./02_implementation.ipynb). It builds
this design in **three passes**: a bad version, a better version, and a
clean final version — so you can *see* why each decision above matters.
